In [1]:
import os

In [2]:
%pwd

'/Users/yashwanthreddygotike/Documents/Projects/Kidney-Disease-Classification-Deep-Learning-Project/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/yashwanthreddygotike/Documents/Projects/Kidney-Disease-Classification-Deep-Learning-Project'

In [5]:
import dagshub
dagshub.init(repo_owner='ycrgotike', repo_name='Kidney-Disease-Classification-Deep-Learning-Project', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as ycrgotike

Initialized MLflow to track repo "ycrgotike/Kidney-Disease-Classification-Deep-Learning-Project"

Repository ycrgotike/Kidney-Disease-Classification-Deep-Learning-Project initialized!

/Users/yashwanthreddygotike/miniconda3/envs/kidney/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🏃 View run skittish-gnat-523 at: https://dagshub.com/ycrgotike/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0/runs/6b37607c0fd542acb1db60a0228ec989
🧪 View experiment at: https://dagshub.com/ycrgotike/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0


In [6]:
import tensorflow as tf


In [7]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

2026-03-09 20:14:37.970380: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-03-09 20:14:37.970508: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-03-09 20:14:37.970515: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-03-09 20:14:37.970878: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-09 20:14:37.971204: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [9]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [10]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone",
            mlflow_uri="https://dagshub.com/ycrgotike/Kidney-Disease-Classification-Deep-Learning-Project.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [11]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [12]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )
            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.keras.log_model(self.model, "model", registered_model_name="EfficientNetB0")
            else:
                mlflow.keras.log_model(self.model, "model")

In [13]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

Found 3732 images belonging to 4 classes.


2026-03-09 20:14:39.120146: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


467/467 [==============================] - 53s 114ms/step - loss: 1.1963 - accuracy: 0.4609


2026/03/09 20:15:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/09 20:15:35 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: /var/folders/vk/rlj4x5g95nq_dq2fhn9j1y4w0000gn/T/tmpyij_77k3/model/data/model/assets


INFO:tensorflow:Assets written to: /var/folders/vk/rlj4x5g95nq_dq2fhn9j1y4w0000gn/T/tmpyij_77k3/model/data/model/assets
Registered model 'EfficientNetB0' already exists. Creating a new version of this model...
2026/03/09 20:16:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: EfficientNetB0, version 2
Created version '2' of model 'EfficientNetB0'.


🏃 View run defiant-fly-975 at: https://dagshub.com/ycrgotike/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0/runs/810b03deadef441fb3402bdb9686c035
🧪 View experiment at: https://dagshub.com/ycrgotike/Kidney-Disease-Classification-Deep-Learning-Project.mlflow/#/experiments/0
